In [28]:
import pandas as pd
import dash
from dash import dcc, html, dash_table
import plotly.express as px
import plotly.graph_objs as go
from dash.dependencies import Input, Output

# Load the data
df = pd.read_csv("student_performance_data.csv")

# Clean the data
df.dropna(subset=["Name"], inplace=True)
df['Marks'] = df['Marks'].fillna(df['Marks'].mean())
df['Attendance(%)'] = df['Attendance(%)'].fillna(df['Attendance(%)'].mean())
df['Logins'] = df['Logins'].fillna(df['Logins'].mean())

# Assign status based on thresholds
def get_status(row):
    if row['Marks'] < 50 or row['Attendance(%)'] < 60:
        return 'Critical'
    elif row['Marks'] < 65 or row['Attendance(%)'] < 70:
        return 'At Risk'
    else:
        return 'On Track'

df['Status'] = df.apply(get_status, axis=1)
color_map = {'On Track': 'green', 'At Risk': 'orange', 'Critical': 'red'}
df['Color'] = df['Status'].map(color_map)

# Create the app
app = dash.Dash(__name__)
app.title = "Student Performance Dashboard"

# Layout
app.layout = html.Div([
    html.H1("📊 Student Performance Dashboard", style={'textAlign': 'center'}),

    # Filters
    html.Div([
        html.Label("Filter by Status:", style={'font-weight': 'bold'}),
        dcc.Dropdown(
            options=[{'label': s, 'value': s} for s in df['Status'].unique()] + [{'label': 'All', 'value': 'All'}],
            value='All',
            id='status-filter',
            clearable=False,
            style={'width': '300px'}
        )
    ], style={'padding': '20px'}),

    # Charts
    html.Div([
        dcc.Graph(id='bar-chart'),
        dcc.Graph(id='before-after-chart'),
        dcc.Graph(id='heatmap')
    ]),

    # Recommendations table
    html.Div([
        html.H3("📋 Personalized Recommendations"),
        dash_table.DataTable(
            id='recommendation-table',
            columns=[
                {'name': 'Name', 'id': 'Name'},
                {'name': 'Marks', 'id': 'Marks'},
                {'name': 'Attendance(%)', 'id': 'Attendance(%)'},
                {'name': 'Logins', 'id': 'Logins'},
                {'name': 'Suggestion', 'id': 'Suggestion'}
            ],
            style_cell={'textAlign': 'center'},
            style_header={'backgroundColor': 'lightgrey', 'fontWeight': 'bold'},
            page_size=10
        )
    ], style={'padding': '0 20px 50px 20px'})
])

# Callback for all components
@app.callback(
    [Output('bar-chart', 'figure'),
     Output('before-after-chart', 'figure'),
     Output('heatmap', 'figure'),
     Output('recommendation-table', 'data')],
    [Input('status-filter', 'value')]
)
def update_dashboard(status):
    filtered_df = df if status == 'All' else df[df['Status'] == status]

    # Chart 1: Marks by Student
    bar_fig = px.bar(
        filtered_df, x='Name', y='Marks', color='Status',
        color_discrete_map=color_map,
        title="Marks by Student",
    )
    bar_fig.update_layout(xaxis_tickangle=-45)

    # Chart 2: Before vs After Intervention
    intervention_df = filtered_df[filtered_df['Status'] != 'On Track'].copy()
    intervention_df = intervention_df.sort_values(by='Marks').head(10)
    intervention_df['Improved Marks'] = intervention_df['Marks'] + 10

    before_after_fig = go.Figure()
    before_after_fig.add_trace(go.Scatter(
        x=intervention_df['Name'], y=intervention_df['Marks'],
        mode='lines+markers', name='Before', line=dict(color='red')
    ))
    before_after_fig.add_trace(go.Scatter(
        x=intervention_df['Name'], y=intervention_df['Improved Marks'],
        mode='lines+markers', name='After', line=dict(color='green')
    ))
    before_after_fig.update_layout(
        title="Before vs After Academic Intervention",
        xaxis_title="Student", yaxis_title="Marks"
    )

    # Chart 3: Correlation Heatmap
    corr = filtered_df[['Marks', 'Attendance(%)', 'Logins']].corr()
    heatmap_fig = px.imshow(
        corr,
        text_auto=True,
        color_continuous_scale='RdBu_r',
        title="Correlation Heatmap"
    )

    # Recommendations Table
    rec_list = []
    for _, row in filtered_df.iterrows():
        if row['Status'] == 'On Track':
            continue
        if row['Attendance(%)'] < 60:
            suggestion = "Counseling & Attendance Monitoring"
        elif row['Logins'] < 15:
            suggestion = "Increase LMS Engagement"
        else:
            suggestion = "Peer Tutoring & Faculty Support"
        rec_list.append({
            'Name': row['Name'],
            'Marks': round(row['Marks'], 1),
            'Attendance(%)': round(row['Attendance(%)'], 1),
            'Logins': int(row['Logins']),
            'Suggestion': suggestion
        })

    return bar_fig, before_after_fig, heatmap_fig, rec_list

# Run the app
if __name__ == '__main__':
    app.run(debug=True)
